# Episode 1 — A character-level tokenizer for news headlines

A language model can't read text. It only does arithmetic on numbers. So before any "AI" happens, we need a way to turn a headline into a list of numbers and back again. That two-way translator is called a **tokenizer**.

In this episode we:

1. **Load and clean** a month of financial news headlines, and look hard at the noise.
2. **Build the simplest possible tokenizer** from scratch: one number per character.
3. **Measure** the dataset and split it into training and validation data *by date*.
4. **Save** everything so Episode 2 (the bigram model) can start straight away.

No machine-learning libraries yet, just plain Python.

## Setup

`ROOT` is the project folder. Adding it to `sys.path` lets this notebook `import market_gpt`, the small package of reusable code that later episodes share. It plays the same role as Composer's autoloader in a PHP project.

In [1]:
import html
import json
import random
import re
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

RAW = ROOT / "data" / "raw" / "news"
PROCESSED = ROOT / "data" / "processed"

---
## Step 1 — Load the headlines

`scripts/download_news.py` saved one file per month in **JSONL** format: one JSON object per line, one article per object. JSONL is handy for big datasets because you can read it line by line, and appending never breaks the file.

In [2]:
articles = []
for path in sorted(RAW.glob("*.jsonl")):
    with path.open(encoding="utf-8") as f:
        month = [json.loads(line) for line in f]
    print(f"{path.name}: {len(month):,} articles")
    articles.extend(month)

print(f"Total: {len(articles):,} articles")

2024-01.jsonl: 20,111 articles
2024-02.jsonl: 23,091 articles
2024-03.jsonl: 21,437 articles


2024-04.jsonl: 23,413 articles
2024-05.jsonl: 26,970 articles


2024-06.jsonl: 16,983 articles
2024-07.jsonl: 22,468 articles


2024-08.jsonl: 24,857 articles
2024-09.jsonl: 17,780 articles


2024-10.jsonl: 25,191 articles


2024-11.jsonl: 23,316 articles
2024-12.jsonl: 16,287 articles
Total: 261,904 articles


One article, as the API gave it to us. We'll only keep `id`, `created_at` and `headline`:

In [3]:
articles[0]

{'id': 36443402,
 'created_at': '2024-01-01T13:05:43Z',
 'updated_at': '2024-01-01T13:05:44Z',
 'symbols': ['AIXI',
  'AMPG',
  'BKYI',
  'CLRO',
  'DATS',
  'FOXO',
  'IFBD',
  'LAES',
  'LIDR',
  'MOB',
  'SAI',
  'WYY'],
 'source': 'benzinga',
 'author': 'Benzinga Insights',
 'headline': "12 Information Technology Stocks Moving In Monday's Pre-Market Session",
 'summary': ' ',
 'url': 'https://www.benzinga.com/trading-ideas/movers/24/01/36443402/12-information-technology-stocks-moving-in-mondays-pre-market-session'}

### Light cleaning

Three small fixes, each for a concrete reason:

- **HTML entities.** The source is a website, so some text arrives HTML-escaped: `&amp;` instead of `&`, `&#39;` instead of `'`. Python's `html.unescape` reverses that. (It's the same as PHP's `html_entity_decode`.)
- **Whitespace.** Collapse runs of spaces, tabs and newlines into a single space. This matters later: we'll put **one headline per line**, so a newline must only ever mean "end of headline".
- **Empty and duplicate headlines** are dropped. Duplicates would teach the model to memorise those exact strings, and a copy sitting in both the training and validation data makes the model look better than it is.

In [4]:
def clean_headline(raw):
    text = html.unescape(raw or "")   # "&amp;" -> "&", "&#39;" -> "'"
    text = " ".join(text.split())     # any run of whitespace -> one space
    return text

example = next(a["headline"] for a in articles if "&amp;" in a["headline"])
print("raw:    ", example)
print("cleaned:", clean_headline(example))

raw:     C&amp;F Financial Coping with Challenges: Bouncing Back with Steady Distribution and Decreasing Investor Rewards
cleaned: C&F Financial Coping with Challenges: Bouncing Back with Steady Distribution and Decreasing Investor Rewards


Now we clean every headline. We go through the articles **oldest first**, so when a headline repeats we keep its first appearance, which is when it was actually news.

Another check runs first: if an article id has already been seen, skip it. Alpaca files articles by the month they were *last updated*, so the same article could turn up in two monthly files.

In [5]:
articles.sort(key=lambda a: a["created_at"])

rows = []
seen_ids, seen_headlines = set(), set()
dropped = Counter()

for a in articles:
    headline = clean_headline(a["headline"])
    if a["id"] in seen_ids:
        dropped["same article twice"] += 1
        continue
    seen_ids.add(a["id"])
    if not headline:
        dropped["empty headline"] += 1
        continue
    if headline in seen_headlines:
        dropped["duplicate headline"] += 1
        continue
    seen_headlines.add(headline)
    rows.append({"id": a["id"], "created_at": a["created_at"], "headline": headline})

print(f"Articles in:   {len(articles):,}")
for reason, n in dropped.items():
    print(f"  dropped, {reason}: {n:,}")
print(f"Headlines out: {len(rows):,}")

Articles in:   261,904
  dropped, duplicate headline: 22,403
Headlines out: 239,501


Which headlines repeated the most? This is our first look at the noise:

In [6]:
repeats = Counter(clean_headline(a["headline"]) for a in articles)
for headline, n in repeats.most_common(8):
    print(f"{n:>3}×  {headline}")

461×  Trading Halt: Halted at 7:50:00 p.m. ET - Trading Halt: Halt News Pending
278×  Trading Halt: Halt status updated at 8:55:00 AM ET: Quotation Resumption: News and Resumption Times
134×  10 Information Technology Stocks With Whale Alerts In Today's Session
128×  10 Information Technology Stocks Whale Activity In Today's Session
123×  10 Consumer Discretionary Stocks Whale Activity In Today's Session
113×  10 Consumer Discretionary Stocks With Whale Alerts In Today's Session
 77×  Trading Halt: Halt status updated at 9:00:00 AM ET: Quotation Resumption: News and Resumption Times
 58×  Trading Halt: Halt status updated at 10:35:00 AM ET: Quotation Resumption: IPO Security - Released for Quotation


**A surprise in the dates.** The downloader asked for 2024, but some articles were *published* years earlier. Alpaca files articles by when they were last *updated*, so an old article edited in 2024 lands in a 2024 file. We'll use `created_at` (the publication date) from here on, because "when was this news?" is what matters.

In [7]:
by_year = Counter(r["created_at"][:4] for r in rows)
for year, n in sorted(by_year.items()):
    print(year, f"{n:>8,}")

2011        1
2012        1
2013        2
2015        1
2016        2
2017        2
2019       11
2020       14
2021       33
2022       27
2023       86
2024  239,321


### The noisy boilerplate

A big share of these "headlines" is generated by machine from templates, not written by a journalist. A few simple patterns (regular expressions, the same `preg_match` patterns you know from PHP) catch the main families. The order matters: the first rule that matches wins.

One family is in between: **analyst ratings** ("UBS Upgrades X to Buy, Raises Price Target to $21"). They follow a template, but each one is a real event that can move a stock, so they get their own label.

In [8]:
KINDS = {
    "trading_halt":       r"^Trading Halt:",
    "movers_roundup":     r"^\d+ .+ Stocks Moving (In|During)",
    "options_activity":   r"(?i)whale|options activity|options trading:|options market|unusual options|smart money|options frenzy",
    "returns_calculator": r"^If You Invested \$|^\$[\d,]+ Invested In|^Here's How Much (\$[\d,]+|You Would)",
    "market_sentiment":   r"^How Is The Market Feeling About",
    "peer_comparison":    r"Versus (Industry )?Peers|Competitors In|In Comparison To Competitors|Industry Competitors",
    "pe_ratio":           r"P/E Ratio|Price Over Earnings",
    "analyst_rating":     r"\b(Maintains|Reiterates|Upgrades|Downgrades|Initiates Coverage On|Assumes)\b.*"
                          r"(Price Target|\b(Buy|Sell|Hold|Neutral|Overweight|Underweight|Outperform|Underperform|Equal-Weight)\b)",
}

def kind_of(headline):
    for kind, pattern in KINDS.items():
        if re.search(pattern, headline):
            return kind
    return "news"

for r in rows:
    r["kind"] = kind_of(r["headline"])

How much of the data is each kind? Look at **characters** as well as headline counts: a character-level model learns from characters, so the character share is what it will "see".

In [9]:
by_kind = Counter(r["kind"] for r in rows)
chars_by_kind = Counter()
for r in rows:
    chars_by_kind[r["kind"]] += len(r["headline"]) + 1   # +1 for the newline
total_chars = sum(chars_by_kind.values())

print(f"{'kind':<20}{'headlines':>10}{'% chars':>10}")
for kind, n in by_kind.most_common():
    print(f"{kind:<20}{n:>10,}{chars_by_kind[kind] / total_chars:>10.1%}")

kind                 headlines   % chars
news                   169,343     76.1%
analyst_rating          62,253     21.6%
options_activity         2,964      0.7%
returns_calculator       2,954      1.0%
market_sentiment           838      0.2%
pe_ratio                   443      0.1%
peer_comparison            337      0.1%
trading_halt               212      0.1%
movers_roundup             157      0.0%


In [10]:
random.seed(0)
for kind in KINDS:
    examples = [r["headline"] for r in rows if r["kind"] == kind]
    print(f"── {kind}")
    for h in random.sample(examples, min(3, len(examples))):
        print("   ", h)

── trading_halt
    Trading Halt: Halted at 7:25:40 a.m. ET - Trading Halt: Halt News Pending
    Trading Halt: Halted at 4:05:00 p.m. ET - Trading Halt: Halt News Pending
    Trading Halt: Halted at 1:38:50 p.m. ET - Trading Halt: Halt News Pending
── movers_roundup
    12 Health Care Stocks Moving In Monday's After-Market Session
    10 Consumer Discretionary Stocks Moving In Thursday's Pre-Market Session
    12 Consumer Staples Stocks Moving In Monday's After-Market Session
── options_activity
    Check Out What Whales Are Doing With COST
    Spotlight on Cava Group: Analyzing the Surge in Options Activity
    Valero Energy Unusual Options Activity For April 15
── returns_calculator
    Here's How Much You Would Have Made Owning Texas Roadhouse Stock In The Last 15 Years
    Here's How Much $1000 Invested In GE Aero 5 Years Ago Would Be Worth Today
    $100 Invested In Lennox Intl 5 Years Ago Would Be Worth This Much Today
── market_sentiment
    How Is The Market Feeling About Tetr

Even "news" still contains template families the rules don't catch: "Analyst Scoreboard: 5 Ratings For …", "What's Going On With … Stock On Monday?". Templates have a long tail, so no set of rules will catch them all.

### Keep it or drop it? The trade-offs

**Reasons to remove the boilerplate:**
- **It's easy to memorise.** "Trading Halt: Halt status updated at 9:55:00 AM ET: Quotation Resumption: News and Resumption Times" is almost the same every time. The model's error goes down because it learned a template, not because it learned anything about language. Part of the training effort goes into copying templates.
- **It flatters the validation score.** If the validation data is full of templates, the model looks better than it really is.
- **Generated samples get dominated by templates**, which makes the "watch random characters turn into headlines" moment in Episode 4 less interesting.

**Reasons to keep it:**
- **It's what the real feed looks like.** In Part 2, a classifier that never saw a trading halt will meet one on its first day.
- **Analyst ratings are real catalysts.** Removing them would delete exactly the events Part 2 wants to study.
- **Every rule is a judgement call with false positives.** The first version of these rules flagged "Rapper 50 Cent … Here's How Much He Earned In Crypto" as a returns calculator.

**What this notebook does: label, don't delete.** Every headline is saved with its `kind`, so each episode can choose with one argument (`exclude_kinds=...`) instead of re-cleaning the data.

**Decision:** for the toy GPT we **drop the machine-made families** and **keep analyst ratings**. The list lives in `market_gpt.data.MACHINE_MADE_KINDS`.

---
## Step 2 — The character-level tokenizer

The idea fits in one sentence: **list every distinct character in the data, and use each character's position in that list as its number.**

That list is the **vocabulary**. Two functions use it:
- `encode(text)` → list of numbers ("token ids")
- `decode(ids)` → text

A *token* is the unit the model reads and predicts. Here one token is one character. That's the simplest choice: small vocabulary, nothing can go wrong, but every headline becomes a *long* list of tokens. Real LLMs use bigger chunks (sub-words); that's the BPE tokenizer later in the series.

First, the whole dataset as one long string, **one headline per line**. The newline character becomes a token too. It means "this headline is over", so a model trained on this text can learn when to stop.

In [11]:
text = "".join(r["headline"] + "\n" for r in rows)
print(f"{len(text):,} characters")
print(text[:300])

22,589,034 characters
Animatronics, Avatars, Afghani And The Marines: Immersive Environment Training
Komunumo: Multi-Generational Households, Economy, and Stocks to Watch
Value Investing for Mothers-in-Law: 'Quality' Requires More than a Good Story
Leaders See a Bright Future For Detroit
Creating Multi-Generational Wealt


The vocabulary is just the set of distinct characters, sorted so it comes out the same on every run:

In [12]:
chars = sorted(set(text))
print(f"{len(chars)} distinct characters:")
print(repr("".join(chars)))

187 distinct characters:
'\n !"#$%&\'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\\]^_abcdefghijklmnopqrstuvwxyz|~¢£¥\xad®°±²·ÀÅÇÉÍÎÓÖØÚÜßàáãäåçèéëíïñòóôõöøúüāČčēğńŌōŘřŞşŪūșʰʳʻ́̈ΕαβεηЄᵈᵗ\u200b‐‑–—•․…\u202a\u202c′\u2060⁵⁸€₹™↯∆≈≤≥ﬀﬁﬃ'


Two lookups. `char_to_id` is a dictionary (a PHP associative array), and for the reverse direction the list itself works because `chars[5]` is the character with id 5:

In [13]:
char_to_id = {ch: i for i, ch in enumerate(chars)}

def encode(s):
    return [char_to_id[ch] for ch in s]

def decode(ids):
    return "".join(chars[i] for i in ids)

### A headline as numbers

In [14]:
headline = "Nvidia Shares Rise 3% After Earnings Beat"
ids = encode(headline)
print(ids)
print()
for ch, i in list(zip(headline, ids))[:12]:
    print(f"{ch!r:>5} -> {i}")
print("   ...")

[47, 86, 73, 68, 73, 65, 1, 52, 72, 65, 82, 69, 83, 1, 51, 73, 83, 69, 1, 20, 6, 1, 34, 70, 84, 69, 82, 1, 38, 65, 82, 78, 73, 78, 71, 83, 1, 35, 69, 65, 84]

  'N' -> 47
  'v' -> 86
  'i' -> 73
  'd' -> 68
  'i' -> 73
  'a' -> 65
  ' ' -> 1
  'S' -> 52
  'h' -> 72
  'a' -> 65
  'r' -> 82
  'e' -> 69
   ...


Notice:
- `' '` is a token like any other, and it's the most common one.
- Capital and small letters are **unrelated numbers** (`'N'` and `'n'`). The model has to learn from data that they are "the same letter".
- The numbers have no meaning in themselves. `'a'` is not "less than" `'b'` in any useful sense. They're just ids, like primary keys in a database table.

The test that matters: decoding gives back exactly what we encoded, for the **entire** dataset:

In [15]:
print(decode(ids))
assert decode(encode(text)) == text
print("Round trip OK for all", f"{len(text):,}", "characters")

Nvidia Shares Rise 3% After Earnings Beat


Round trip OK for all 22,589,034 characters


### The same thing, packaged for reuse

Those few lines live in [`market_gpt/tokenizer.py`](../market_gpt/tokenizer.py) as a `CharTokenizer` class, with `save()` and `load()` added so Episode 2 can import it. Same logic, same numbers:

In [16]:
from market_gpt.tokenizer import CharTokenizer

tok = CharTokenizer.from_text(text)
assert tok.chars == chars and tok.encode(headline) == ids
print("vocab_size:", tok.vocab_size)

vocab_size: 187


### The weakness: characters it has never seen

The vocabulary only knows characters that appeared in the data. Give it something new and it has no number to use:

In [17]:
try:
    tok.encode("Tesla to the moon 🚀")
except ValueError as e:
    print("ValueError:", e)

ValueError: Character '🚀' is not in the vocabulary


Real tokenizers avoid this by working on **bytes**. Every possible text is some sequence of the 256 possible byte values, so nothing is ever "unknown". That's where the BPE episode will start.

(For the same reason, we build the vocabulary from *all* the headlines, including the validation ones. The vocabulary is only the alphabet, so the model learns nothing about the validation data from it. Building it from the training data alone would just make validation crash on a rare character.)

---
## Step 3 — Stats

In [18]:
lengths = [len(r["headline"]) for r in rows]
print(f"Headlines:        {len(rows):,}")
print(f"Characters:       {len(text):,}  (= tokens, since 1 character = 1 token)")
print(f"Vocabulary size:  {tok.vocab_size}")
print(f"Headline length:  shortest {min(lengths)}, median {sorted(lengths)[len(lengths)//2]}, longest {max(lengths)} characters")

Headlines:        239,501
Characters:       22,589,034  (= tokens, since 1 character = 1 token)
Vocabulary size:  187
Headline length:  shortest 9, median 82, longest 512 characters


The median headline is about 80 tokens long. A model has to predict all 80, one at a time, to write a single headline. Keep that number in mind for the BPE episode.

### Most and least common characters

In [19]:
NAMES = {" ": "space", "\n": "newline", "​": "zero-width space", "\xa0": "no-break space"}
def show(ch):
    return NAMES.get(ch, repr(ch))

counts = Counter(text)
print("Most common:")
for ch, n in counts.most_common(12):
    print(f"  {show(ch):<10}{n:>10,}  {n / len(text):6.2%}")

Most common:
  space      3,173,467  14.05%
  'e'        1,791,644   7.93%
  'a'        1,251,844   5.54%
  't'        1,192,802   5.28%
  'i'        1,191,244   5.27%
  'n'        1,180,587   5.23%
  'r'        1,115,282   4.94%
  's'        1,089,457   4.82%
  'o'        1,082,763   4.79%
  'l'          569,594   2.52%
  'c'          474,589   2.10%
  'u'          411,900   1.82%


In [20]:
print("Least common (with a headline that uses each one):")
for ch, n in sorted(counts.items(), key=lambda kv: kv[1])[:12]:
    example = next(r["headline"] for r in rows if ch in r["headline"])
    print(f"  {show(ch):<20}{n:>3}   {example[:75]}")

Least common (with a headline that uses each one):
  'ń'                   1   Mainz Biomed And TestDNA Drive CRC Innovation At The 10th Gdańsk Gastroente
  '′'                   1   Former Trump Challenger Chris Christie Not Ruling Out Running As An Indepen
  '^'                   1   Genenta Science Has Dosed First Of Three Patients In Cohort 8 (Temferon At 
  'Ū'                   1   Hawaiian Electric Company Announces Ameresco Award To Develop The 'Ūkiu Ene
  'õ'                   1   Tellurian Board Decided Not To Renew Or Extend CEO Octávio Simões' Employme
  'η'                   1   Accenture Agreed To Acquire Arηs Group; No Terms Disclosed
  '‐'                   1   Sonoma Pharmaceuticals Reported A Recent Publication In The Journal Neurour
  'ʳ'                   1   Cytokinetics Says It Presented Additional 48-Week Data From FOREST-HCM, The
  'ᵈ'                   1   Cytokinetics Says It Presented Additional 48-Week Data From FOREST-HCM, The
  '↯'                   1   'C

The rare characters are a long tail: accents in names, Greek letters, superscripts, symbols like `↯`. Look back at the vocabulary too: it contains **invisible** characters (`\u200b` zero-width space, `\xad` soft hyphen, `\u202a`/`\u202c` text-direction marks) that came along with copy-pasted text. Each one gets its own slot in the vocabulary even though the model will barely see it. With a handful of examples it can't learn much about them, and that's fine.

In [21]:
ascii_share = sum(n for ch, n in counts.items() if ord(ch) < 128) / len(text)
print(f"{ascii_share:.3%} of all characters are plain ASCII")

99.981% of all characters are plain ASCII


### Train / validation split, by date

We hold back some data the model never trains on (the **validation set**) to measure whether it learned general patterns or just memorised the training text.

**Why split by date instead of at random?** Because the model will be used on *tomorrow's* headlines, not on a random sample of last month's. A random split puts Tuesday's headlines in training and Monday's in validation. News comes in waves: the same story gets re-reported, updated and corrected within hours. So a random split leaks near-copies of the validation headlines into training, and the score looks better than it really is.

This is the same trap as the trading project's backtest: test on data the system effectively already saw, and the results look great until reality arrives. So: **train on the past, validate on the future.**

We take whole days, starting from the newest and working backwards, until at least 10% of the headlines are in validation. Everything before that is training data. (Whole days, because cutting in the middle of a day would split a single news wave across both sides.)

In [22]:
per_day = Counter(r["created_at"][:10] for r in rows)

in_val = 0
for day in sorted(per_day, reverse=True):   # newest day first
    in_val += per_day[day]
    if in_val >= 0.1 * len(rows):
        val_start = day                      # the first day of validation
        break

for r in rows:
    r["split"] = "train" if r["created_at"][:10] < val_start else "val"

train_rows = [r for r in rows if r["split"] == "train"]
val_rows = [r for r in rows if r["split"] == "val"]

for name, part in [("train", train_rows), ("val", val_rows)]:
    n_chars = sum(len(r["headline"]) + 1 for r in part)
    print(f"{name:<6}{len(part):>7,} headlines {n_chars:>11,} characters   "
          f"{part[0]['created_at'][:10]} → {part[-1]['created_at'][:10]}")

train 214,864 headlines  20,157,494 characters   2011-11-16 → 2024-11-12
val    24,637 headlines   2,431,540 characters   2024-11-13 → 2024-12-31


(The training set "starts" years before 2024 because of those old, recently-updated articles. It's harmless: they're in the past either way.)

**Does the leak actually happen here?** Let's measure it. For each validation headline, look for a training headline that **starts with the same 8 words and was published within 24 hours of it**, which is a rough sign of "the same story, re-posted or updated". We do this for our date split and for a random split of the same size.

In [23]:
from datetime import datetime, timedelta

def first_words(h, n=8):
    return " ".join(h.split()[:n])

def published(r):
    return datetime.fromisoformat(r["created_at"].replace("Z", "+00:00"))

def near_copies(train, val):
    # Index training headlines by their first words: {first 8 words: [publication times]}
    index = {}
    for r in train:
        index.setdefault(first_words(r["headline"]), []).append(published(r))
    # Validation headlines that have a training twin within 24 hours
    leaks = []
    for r in val:
        for t in index.get(first_words(r["headline"]), []):
            if abs(published(r) - t) < timedelta(hours=24):
                leaks.append(r)
                break
    return leaks

shuffled = rows[:]
random.seed(0)
random.shuffle(shuffled)
rand_train, rand_val = shuffled[:len(train_rows)], shuffled[len(train_rows):]

by_date = near_copies(train_rows, val_rows)
by_random = near_copies(rand_train, rand_val)
print("Validation headlines with a same-day near-copy in training:")
print(f"  split by date:  {len(by_date):>4}  ({len(by_date) / len(val_rows):.1%})")
print(f"  random split:   {len(by_random):>4}  ({len(by_random) / len(rand_val):.1%})")

Validation headlines with a same-day near-copy in training:
  split by date:     2  (0.0%)
  random split:    345  (1.4%)


A few of the leaks from the random split, each next to its training twin:

In [24]:
def twin(v, train):
    # The training headline with the same first words, published closest in time to v
    same_start = [t for t in train if first_words(t["headline"]) == first_words(v["headline"])]
    return min(same_start, key=lambda t: abs(published(t) - published(v)))

random.seed(1)
for v in random.sample(by_random, 4):
    t = twin(v, rand_train)
    print("train:", t["created_at"][:16], t["headline"][:90])
    print("val:  ", v["created_at"][:16], v["headline"][:90])
    print()

train: 2024-12-04T17:49 President-Elect Donald J. Trump Posts On Truth "I am delighted to announce the nomination 
val:   2024-12-04T17:53 President-Elect Donald J. Trump Posts On Truth "I am proud to announce the appointment of 



train: 2024-04-10T16:30 Here's How Much You Would Have Made Owning e.l.f. Beauty Stock In The Last 5 Years
val:   2024-04-10T17:30 Here's How Much You Would Have Made Owning PVH Stock In The Last 20 Years



train: 2024-04-17T13:48 Edible Garden Shares Halted On Circuit Breaker To The Upside, Stock Now Up 65.5%
val:   2024-04-17T14:06 Edible Garden Shares Halted On Circuit Breaker To The Upside, Stock Now Up 144.2%



train: 2024-04-18T21:31 Here's How Much You Would Have Made Owning ATI Stock In The Last 5 Years
val:   2024-04-18T20:30 Here's How Much You Would Have Made Owning T-Mobile US Stock In The Last 5 Years



The date split leaks almost nothing; only the boundary day can have twins on both sides. The random split leaks more than ten times as many. On headlines that's still only a percent or two of the validation set, so it inflates the score a little, not a lot. Prices are different: neighbouring minutes are almost identical, so a random split leaks almost everything. That's how a backtest ends up looking brilliant.

In [25]:
only_in_val = set("".join(r["headline"] for r in val_rows)) - set("".join(r["headline"] for r in train_rows))
print("Characters that appear only in validation:", [show(c) for c in sorted(only_in_val)])

Characters that appear only in validation: ["'Ú'", "'Ō'", "'ε'", "'․'", "'ﬃ'"]


If we had built the vocabulary from the training headlines only, `encode()` would crash on the validation set because of these characters. That's why the vocabulary is built from all the headlines (see Step 2).

---
## Step 4 — Save for Episode 2

Two files in `data/processed/` (git-ignored, because re-running this notebook rebuilds them):

- `headlines.jsonl`: one cleaned headline per line, with its date, `kind` and `split`.
- `char_tokenizer.json`: the vocabulary, the only thing needed to rebuild the tokenizer.

In [26]:
PROCESSED.mkdir(parents=True, exist_ok=True)

with (PROCESSED / "headlines.jsonl").open("w", encoding="utf-8") as f:
    for r in rows:
        record = {"id": r["id"], "created_at": r["created_at"], "kind": r["kind"],
                  "split": r["split"], "headline": r["headline"]}
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

tok.save(PROCESSED / "char_tokenizer.json")

for path in sorted(PROCESSED.iterdir()):
    print(f"{path.name:<22}{path.stat().st_size / 1e6:6.2f} MB")

char_tokenizer.json     0.00 MB
headlines.jsonl        48.18 MB


This is how Episode 2 will load it. `exclude_kinds=MACHINE_MADE_KINDS` is where the boilerplate decision gets applied:

In [27]:
from market_gpt.data import MACHINE_MADE_KINDS, TOKENIZER, as_text, load_headlines

tok = CharTokenizer.load(TOKENIZER)
train = load_headlines("train")
val = load_headlines("val")
print(f"train: {len(train):,} headlines   val: {len(val):,} headlines")

train_clean = load_headlines("train", exclude_kinds=MACHINE_MADE_KINDS)
print(f"train without machine-made boilerplate: {len(train_clean):,} headlines")

sample = as_text(val[:2])
print(sample)
print(tok.encode(sample)[:30], "...")

train: 214,864 headlines   val: 24,637 headlines


train without machine-made boilerplate: 207,604 headlines
Sustainable Investment Under Trump: 'Performance Matters Far More Than Politics,' JPMorgan Analyst Says
Chamath Palihapitiya Says Cut The Federal Budget By Trillions And Then Do This With A 'Fraction' Of The Money To 'Make America Great Again'

[52, 85, 83, 84, 65, 73, 78, 65, 66, 76, 69, 1, 42, 78, 86, 69, 83, 84, 77, 69, 78, 84, 1, 54, 78, 68, 69, 82, 1, 53] ...


---
## Wrap-up

- A **tokenizer** turns text into numbers and back. Ours gives every character its own id: a vocabulary of under 200 characters, `encode()` and `decode()`, and a guaranteed round trip.
- Character-level is simple and never surprised by spelling, but it's **long**: about 80 tokens per headline. It also knows nothing about characters outside its vocabulary.
- Close to a third of the headlines are **templates**, mostly analyst ratings. We labelled them instead of deleting them.
- We split **by date**, because the model must work on the future, and a random split leaks same-day near-copies into validation.
- Deduplication alone removed most trading halts. They repeat word for word, so the boilerplate rules matter less than the June-only numbers suggested.

**Next, Episode 2:** the bigram model, the simplest thing that can "predict the next character".